In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/빅데이터의사결정분석/Project

/content/drive/MyDrive/빅데이터의사결정분석/Project


In [3]:
!pip install -q contractions

In [1]:
# DataFrame 전처리 라이브러리
import pandas as pd
import numpy as np
import tensorflow as tf

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 텍스트 데이터 전처리
from bs4 import BeautifulSoup
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import contractions
from collections import Counter
import nltk
import matplotlib.pyplot as plt

# 텍스트 데이터 임베딩
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# sklearn 정규화, 평가지표
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# sklearn 머신러닝 모델
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

In [2]:
# 경고 문구 해제
import warnings
warnings.filterwarnings(action='ignore')

# **Utils**

In [3]:
def preprocess_text_lemmatized(text, normalization_dict=None):

    '''
    1. HTML 태그 제거
    2. 특정 단어 정규화 (옵션)
    3. 특수 문자 제거
    4. 모든 문자를 소문자로 변환
    5. 텍스트를 토큰화 및 불용어 제거
    6. 표제어 추출 및 원형 복원
    7. 표제어 추출 리스트를 String 형식으로 변경
    '''

  # HTML 태그 제거
    text_no_html = BeautifulSoup(text, "html.parser").get_text()

  # 특정 단어 정규화 (옵션)
    if normalization_dict:
        for old, new in normalization_dict.items():
            text_no_html = text_no_html.replace(old, new)

  # 특수 문자 제거
    text_no_specials = re.sub(r'[^a-zA-Z]', ' ', text_no_html)

  # 여러 공백을 하나로 축소 및 공백 정리
  # 띄어쓰기가 여럿 있으면, 띄어쓰기 하나로 만들어줌
    text_clean = re.sub(r'\s+', ' ', text_no_specials).strip()

  # 모든 문자를 소문자로 변환
    texts_lower = text_clean.lower()

  # 텍스트를 토큰화 및 불용어 제거
    tokens = word_tokenize(texts_lower)
    filtered_tokens = [word for word in tokens if word not in stopwords.words('english')]

  # lemmatizer Class 호출
    lemmatizer = WordNetLemmatizer()

  # 리스트 형태의 출력을 DataFrame에 저장할 수 있도록 String으로 변환해줌
    lemmatized_tokens = ' '.join([lemmatizer.lemmatize(word) for word in filtered_tokens])

    return lemmatized_tokens

In [4]:
def filter_text_by_frequency(df, column_name, min_frequency, out_column_name):

    '''
    1. 데이터프레임의 모든 텍스트를 하나의 문자열로 합치기
    2. 토큰화하여 단어의 빈도 계산하기
    3. 특정 빈도 이하의 단어 제거하기
    4. 필터링된 단어들로 새로운 텍스트 집합 생성
    5. 필터링된 텍스트를 새로운 데이터프레임 컬럼에 저장
    '''

  # 1. 데이터프레임의 모든 텍스트를 하나의 문자열로 합치기
    all_text = ' '.join(df[column_name])

  # 2. 토큰화하여 단어의 빈도 계산하기
    tokens = word_tokenize(all_text)
    word_counts = Counter(tokens)

  # 3. 특정 빈도 이하의 단어 제거하기
    filtered_words = [word for word, count in word_counts.items() if count > min_frequency]

  # 4. 필터링된 단어들로 새로운 텍스트 집합 생성
    filtered_texts = [' '.join(word for word in word_tokenize(text) if word in filtered_words) for text in df[column_name]]

  # 필터링된 텍스트를 새로운 데이터프레임 컬럼에 저장
    df[out_column_name] = filtered_texts

    return df

In [5]:
def tokenize_and_create_embedding_matrix(train_text, test_text, max_words=10000, max_sequence_length=300):

    '''
    1. 토크나이저 초기화 (OOV 토큰 포함)
    2. 훈련 데이터에 토크나이저를 적용해 학습
    3. 텍스트 데이터를 시퀀스로 변환
    4. 시퀀스 패딩 처리 (post 방식 사용)
    5. Word2Vec 모델로 텍스트간의 관계를 학습
    '''

  # 토크나이저 초기화 (OOV 토큰 포함)
    tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')

  # 훈련 데이터에 토크나이저를 적용해 학습
    tokenizer.fit_on_texts(train_text)

  # 텍스트 데이터를 시퀀스로 변환
    train_sequences = tokenizer.texts_to_sequences(train_text)
    test_sequences = tokenizer.texts_to_sequences(test_text)

  # 시퀀스 패딩 처리 (post 방식 사용)
    padded_train_text = pad_sequences(train_sequences, maxlen=max_sequence_length, padding='post')
    padded_test_text = pad_sequences(test_sequences, maxlen=max_sequence_length, padding='post')

  # 단어 인덱스 맵 생성 및 임베딩 행렬 준비
  # 임베딩 차원을 300으로 가정
    word_index = tokenizer.word_index
    total_words = min(max_words, len(word_index))
    embedding_matrix = np.zeros((total_words + 1, 300))

  # Word2Vec 모델에서 단어에 해당하는 벡터를 임베딩 행렬에 할당
    tokenized_x_train = [word_tokenize(sentence) for sentence in train_text]
    tokenized_x_test = [word_tokenize(sentence) for sentence in test_text]

  # Word2Vec 모델 훈련
    w2v = Word2Vec(sentences=tokenized_x_train, vector_size=300, window=5, min_count=1, workers=4)

  # 각 embedding의 평균을 구함
    def get_average_embedding(tokenized_text, w2v):
        embeddings = [w2v.wv[word] for word in tokenized_text if word in w2v.wv]
        if embeddings:
            return np.mean(embeddings, axis=0)
        else:
            return np.zeros(300)

    train_embeddings = np.array([get_average_embedding(sentence, w2v) for sentence in tokenized_x_train])
    test_embeddings = np.array([get_average_embedding(sentence, w2v) for sentence in tokenized_x_test])

  # 패딩 처리된 텍스트 데이터와 임베딩 행렬, 사용된 총 단어 수, 시퀀스 최대 길이 반환
    return train_embeddings, test_embeddings, embedding_matrix, total_words, max_sequence_length

# **데이터 전처리**

In [6]:
# NLTK의 불용어 리스트 다운로드
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to ~/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [7]:
review = pd.read_csv('project_review_helpful.csv')

print(review.shape)
review.head()

(24838, 21)


In [8]:
# 전처리 함수 적용
review['lemmatized_text'] = review['review_text'].progress_apply(preprocess_text_lemmatized)
review.head()

  0%|          | 0/24838 [00:00<?, ?it/s]

100%|██████████| 24838/24838 [01:19<00:00, 313.31it/s]


In [9]:
# stemmting, lemmatize 한 단어들 중에서 5개 이하로 언급된 단어 제외
filter_text_by_frequency(review, 'lemmatized_text', min_frequency=5, out_column_name='filter_lemmatized_text')

In [10]:
# 훈련 및 테스트 데이터 분할
train, test = train_test_split(review, test_size=0.2, random_state=42,  stratify= review["fake"])

train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

y_train = np.array(train['fake'])
y_test = np.array(test['fake'])

# **Word2vec을 이용해 Text 임베딩의 평균을 추출**


In [11]:
train_embedding_matrix, test_embedding_matrix, embedding_matrix, total_words, max_sequence_length = tokenize_and_create_embedding_matrix(train['filter_lemmatized_text'], test['filter_lemmatized_text'], max_words=10000, max_sequence_length=300)

In [12]:
train_embedding_matrix.shape

In [13]:
test_embedding_matrix.shape

In [14]:
# 추출한 embedding을 데이터 프레임으로 만듦
train_embedding = pd.DataFrame(train_embedding_matrix)
test_embedding = pd.DataFrame(test_embedding_matrix)

# **숫자형 변수랑 concat**

In [15]:
# 숫자형 변수들을 저장

review_num_feature = ['experience', 'depth','reputation', 'text_structure', 'style']

X_train = train[review_num_feature]
X_test = test[review_num_feature]

In [16]:
Minmax = MinMaxScaler()
X_train = Minmax.fit_transform(X_train)
X_test = Minmax.transform(X_test)

X_train = pd.DataFrame(X_train, columns=review_num_feature)
X_test = pd.DataFrame(X_test, columns=review_num_feature)

In [17]:
X_train

In [18]:
# 최종 데이터셋 형성
X_train = pd.concat([X_train, train_embedding], axis=1)
X_test = pd.concat([X_test, test_embedding], axis=1)

In [19]:
# sklearn 모델에 투입하기 위해서 column의 형식을 string으로 만들어줌
X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

In [20]:
# train dataset 확인
X_train

# **머신러닝 모델 학습**

In [21]:
# 모델 정의
models = {
    'SVC': SVC(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'AdaBoost': AdaBoostClassifier(),
    'XGBoost': XGBClassifier(),
    'Logistic Regression': LogisticRegression(),
    'SGD Classifier': SGDClassifier(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB()
}

# 결과를 저장할 데이터프레임 초기화
results = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score'])

# 각 모델에 대해 학습 및 평가
for name, model in tqdm(models.items()):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # 성능 평가 지표 계산
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # 데이터프레임에 결과 추가
    result_df = pd.DataFrame([[name, accuracy, precision, recall, f1]],
                             columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score'])
    results = pd.concat([results, result_df], ignore_index=True)

# 결과 출력
results

100%|██████████| 10/10 [05:10<00:00, 31.02s/it]


In [22]:
results

In [32]:
results.sort_values(by='F1-score', ascending=False).reset_index(drop=True)

## **평가지표 F1-Score가 가장 높은 순: Gradient Boosting > Random Forest > XGBoost**